# 07 — Live Market Data (Yahoo/yfinance)

> **This notebook is optional and not deterministic.** It can make a live
> network request through the optional `yfinance` dependency. If you don't
> have network access or the `market` extra installed
> (`pip install abaquant[market]`), the cell below will raise a
> `MarketDataError`, which is caught and reported cleanly.

**What it shows:**
- Constructing a `MarketTicker` backed by the Yahoo Finance provider, with
  disk caching for financial statements.
- Retrieving (or reusing a cached) annual statement snapshot.
- Reading major line items and running the credit-proxy assessment.
- Optional visualization of the balance sheet and credit score.


## Setup


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

def _ensure_abaquant_importable():
    try:
        import abaquant  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        src = candidate / "src"
        if (src / "abaquant" / "__init__.py").exists():
            sys.path.insert(0, str(src))
            return
    raise ModuleNotFoundError(
        "Could not import abaquant. Run this notebook from within the AbaQuant "
        "repository (or pip install abaquant)."
    )

_ensure_abaquant_importable()
import abaquant
print(f"AbaQuant version: {abaquant.__version__}")

In [ ]:
from pathlib import Path

from abaquant.marketdata import get_ticker
from abaquant.marketdata.errors import MarketDataError
from abaquant.visualization import VisualizationError

## Build a live Yahoo-backed ticker

Financial statements are cached to disk under `.qa_example_cache/` so
repeated runs don't re-request the same data.


In [ ]:
SYMBOL = "NVDA"

ticker = get_ticker(
    SYMBOL,
    provider="yahoo",
    financial_cache="disk",
    cache_directory=Path(".qa_example_cache"),
)

## Retrieve or reuse the annual statement snapshot


In [ ]:
def run_live_example():
    ticker.financials.snapshot(period="annual", refresh_policy="if_stale", max_age_days=7)
    assessment = ticker.credit.assess_from_financials(period="annual")
    summary = {
        "symbol": ticker.symbol,
        "total_debt": ticker.financials.total_debt(period="annual"),
        "ebitda": ticker.financials.ebitda(period="annual"),
        "operating_cash_flow": ticker.financials.operating_cash_flow(period="annual"),
        "synthetic_score": assessment.synthetic_credit_proxy_score,
        "synthetic_band": assessment.synthetic_credit_proxy_band,
    }
    return assessment, summary


try:
    assessment, summary = run_live_example()
    for key, value in summary.items():
        print(f"{key:24s}: {value}")
except MarketDataError as exc:
    print("Live Yahoo example skipped (no network access or missing 'market' extra).")
    print(exc)

## Optional: visualize the balance sheet and credit score


In [ ]:
try:
    figures = {
        "balance_sheet": ticker.financials.visualize(statement="balance_sheet"),
        "credit_score": assessment.visualize(chart="score"),
    }
    print(f"Created {len(figures)} figures: {list(figures)}")
except NameError:
    print("Skipped: the live retrieval cell above did not complete.")
except VisualizationError as exc:
    print(f"Visualization skipped (optional dependency missing): {exc}")

## Takeaway

Everything else in this notebook collection uses deterministic, offline
fixtures on purpose — see notebook **06 — Market Data (Offline)** for the
same workflow with reproducible, network-free data.
